# riemPrim NNI results

In [ ]:
import json
from pathlib import Path
from motion_primitives.paths import PROJECT_ROOT

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# NNI trial root containing diagnostics.json files and destination for tables/figures.
results_dir = PROJECT_ROOT / "results/ReachGrasp/riemannian/evaluate_riemPrim_params/nni"
output_dir = PROJECT_ROOT / "results/ReachGrasp/riemannian/plot_riemPrim_nni_results"

rows = []
run_params = {}
for path in sorted(results_dir.glob("*/diagnostics.json")):
    diagnostics = json.loads(path.read_text())
    run = path.parent.name
    run_params[run] = json.loads((path.parent / "run_config.json").read_text())["params"]
    rows.append({
        "run": run,
        "n_primitives": diagnostics["n_primitives"],
        "mean_endpoint_error": diagnostics["mean_endpoint_error"],
    })

results = pd.DataFrame(rows)
output_dir.mkdir(parents=True, exist_ok=True)
results[["n_primitives", "mean_endpoint_error"]].to_csv(output_dir / "riemPrim_nni_results.csv", index=False)
results

In [ ]:
best_by_count = results.loc[results.groupby("n_primitives")["mean_endpoint_error"].idxmin()].sort_values("n_primitives")
pareto = best_by_count[best_by_count["mean_endpoint_error"] == best_by_count["mean_endpoint_error"].cummin()].copy()
x = (pareto["n_primitives"] - pareto["n_primitives"].min()) / (pareto["n_primitives"].max() - pareto["n_primitives"].min())
y = (pareto["mean_endpoint_error"] - pareto["mean_endpoint_error"].min()) / (pareto["mean_endpoint_error"].max() - pareto["mean_endpoint_error"].min())
elbow = pareto.iloc[np.abs(x + y - 1).argmax()]
elbow_result = {
    "run": elbow["run"],
    "n_primitives": int(elbow["n_primitives"]),
    "mean_endpoint_error": float(elbow["mean_endpoint_error"]),
    "parameters": run_params[elbow["run"]],
}
(output_dir / "elbow_parameters.json").write_text(json.dumps(elbow_result, indent=2) + "\n")

plt.figure(figsize=(9, 6))
plt.scatter(results["n_primitives"], results["mean_endpoint_error"])
plt.scatter(elbow["n_primitives"], elbow["mean_endpoint_error"], color="red", marker="x", s=100, label="elbow")
plt.xlabel("number of primitives")
plt.ylabel("mean endpoint error")
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / "error_vs_primitives_mean_endpoint_error.png", dpi=180)
plt.show()